# Module 1 — Local LLM Basics

Companion notebook to [`docs/modules/01_local_llm_systems_thinking.md`](../docs/modules/01_local_llm_systems_thinking.md).

This notebook is meant to be **run**, not just read. It:

1. Derives the memory-budget mental model with real numbers (weights + KV cache).
2. Shows why `tiktoken`-style word-count heuristics are only for rough estimation, never for budgeting.
3. Runs Labs 1.1–1.3 against a local Ollama server **if one is available**, and otherwise
   prints exactly what's missing rather than faking results.

Run with: `uv run jupyter lab` from the repo root, then open this file.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODULE_01_SCRIPTS = REPO_ROOT / "scripts" / "module_01"
sys.path.insert(0, str(MODULE_01_SCRIPTS))

print(f"Repo root: {REPO_ROOT}")
print(f"Module 1 scripts on path: {MODULE_01_SCRIPTS.exists()}")

Repo root: /Users/bhakti/workspace/local_ai_app
Module 1 scripts on path: True


## 1. The memory budget formula, with real numbers

```text
total_memory ≈ model_weights + KV_cache + runtime_overhead + compute_buffers + app_memory + OS_memory
```

Let's make the two most important terms concrete for a Qwen2.5-7B-style model:
`n_layers=28`, `n_kv_heads=4`, `head_dim=128` (grouped-query attention — note we use
`n_kv_heads`, not the full attention head count).

In [2]:
def weights_gib(n_params_billion: float, bits_per_param: float) -> float:
    """Approximate resident memory for model weights, in GiB."""
    bytes_per_param = bits_per_param / 8
    total_bytes = n_params_billion * 1e9 * bytes_per_param
    return total_bytes / (1024 ** 3)


def kv_cache_gib(
    n_layers: int,
    n_kv_heads: int,
    head_dim: int,
    context_tokens: int,
    bytes_per_element: float = 2.0,  # FP16
    concurrent_sequences: int = 1,
) -> float:
    """Approximate KV cache memory, in GiB. Mirrors the formula in the theory doc."""
    elements_per_token = 2 * n_layers * n_kv_heads * head_dim  # 2 = K and V
    total_bytes = elements_per_token * context_tokens * bytes_per_element * concurrent_sequences
    return total_bytes / (1024 ** 3)


quant_bits = {"FP16": 16.0, "Q8_0": 8.5, "Q6_K": 6.6, "Q5_K_M": 5.7, "Q4_K_M": 4.8}
print("Weight footprint for a 7B model by quantization:")
for name, bits in quant_bits.items():
    print(f"  {name:8s} -> {weights_gib(7, bits):.2f} GiB")

Weight footprint for a 7B model by quantization:
  FP16     -> 13.04 GiB
  Q8_0     -> 6.93 GiB
  Q6_K     -> 5.38 GiB
  Q5_K_M   -> 4.64 GiB
  Q4_K_M   -> 3.91 GiB


In [3]:
print("KV cache footprint for that model at growing context lengths (single sequence, FP16 cache):")
for ctx in [4_096, 8_192, 32_768, 128_000]:
    gib = kv_cache_gib(n_layers=28, n_kv_heads=4, head_dim=128, context_tokens=ctx)
    print(f"  context={ctx:>7,} tokens -> {gib:.3f} GiB")

print()
print("The punchline: weights are ~fixed once you pick a quantization; KV cache is the")
print("term that blows up with context length and concurrency. A model that 'fits' at")
print("4K context can still be unusable at 32K, or under 4 concurrent requests.")

KV cache footprint for that model at growing context lengths (single sequence, FP16 cache):
  context=  4,096 tokens -> 0.219 GiB
  context=  8,192 tokens -> 0.438 GiB
  context= 32,768 tokens -> 1.750 GiB
  context=128,000 tokens -> 6.836 GiB

The punchline: weights are ~fixed once you pick a quantization; KV cache is the
term that blows up with context length and concurrency. A model that 'fits' at
4K context can still be unusable at 32K, or under 4 concurrent requests.


In [4]:
print("Concurrency multiplies the KV-cache term directly:")
for n_concurrent in [1, 2, 4, 8]:
    gib = kv_cache_gib(
        n_layers=28, n_kv_heads=4, head_dim=128, context_tokens=8_192,
        concurrent_sequences=n_concurrent,
    )
    print(f"  {n_concurrent} concurrent requests at 8K context -> {gib:.3f} GiB KV cache")

Concurrency multiplies the KV-cache term directly:
  1 concurrent requests at 8K context -> 0.438 GiB KV cache
  2 concurrent requests at 8K context -> 0.875 GiB KV cache
  4 concurrent requests at 8K context -> 1.750 GiB KV cache
  8 concurrent requests at 8K context -> 3.500 GiB KV cache


## 2. Token counting: heuristic vs exact

`token_estimate.estimate_tokens_heuristic` is deliberately *not* named `count_tokens` —
it exists only to build stress-test prompts of an approximate target length (Lab 1.2).
It must never be used to make an admission-control or budgeting decision.

In [5]:
from token_estimate import estimate_tokens_heuristic, words_for_target_tokens, HFTokenizerCounter, TokenizerUnavailable

sample = "Local LLM engineering is systems engineering under a hard, visible memory ceiling."
print(f"Heuristic estimate: {estimate_tokens_heuristic(sample)} tokens for {len(sample.split())} words")

# Exact counting requires a real tokenizer. Try it; fall back honestly if unavailable.
try:
    counter = HFTokenizerCounter("Qwen/Qwen2.5-1.5B-Instruct")
    exact = counter.count(sample)
    print(f"Exact tokenizer count (Qwen2.5): {exact} tokens")
except TokenizerUnavailable as exc:
    print(f"Exact tokenizer unavailable in this environment: {exc}")
    print("This is expected if `transformers` isn't installed or there's no network ")
    print("access to download the tokenizer files. Install with `uv add transformers` ")
    print("to try this locally.")

Heuristic estimate: 16 tokens for 12 words
Exact tokenizer unavailable in this environment: Could not load tokenizer for 'Qwen/Qwen2.5-1.5B-Instruct'. Install `transformers` and ensure the model id is a valid Hugging Face repo, or fall back to runtime-reported counts.
This is expected if `transformers` isn't installed or there's no network 
access to download the tokenizer files. Install with `uv add transformers` 
to try this locally.


## 3. Running the labs against a live Ollama server

The cells below check for a reachable local Ollama server first. On the machine this
course was authored on, **Ollama is not installed** (see `PROGRESS.md` and
`reports/module_01_local_llm_observations.md`), so these cells will print a clear
skip message. Once Module 2 sets up Ollama and pulls a few models, re-run this section
to get real measurements.

In [6]:
from ollama_probe import is_ollama_available, list_local_models

if is_ollama_available():
    print("Ollama is reachable. Local models:")
    for m in list_local_models():
        print(f"  - {m}")
else:
    print("Ollama is NOT reachable at http://localhost:11434.")
    print("Install it (Module 2: `brew install ollama` or the macOS app), start it,")
    print("pull at least one model (`ollama pull qwen2.5:1.5b`), and re-run this cell.")

Ollama is NOT reachable at http://localhost:11434.
Install it (Module 2: `brew install ollama` or the macOS app), start it,
pull at least one model (`ollama pull qwen2.5:1.5b`), and re-run this cell.


In [7]:
# Lab 1.1 — multi-model comparison. Only runs meaningfully if models are pulled locally.
from lab_1_1_multi_model_run import run_lab, rows_to_markdown_table, DEFAULT_PROMPT
from IPython.display import Markdown, display

if is_ollama_available():
    available = set(list_local_models())
    candidates = [m for m in ["qwen2.5:1.5b", "qwen2.5:3b", "qwen2.5:7b"] if m in available]
    if candidates:
        rows = run_lab(candidates, DEFAULT_PROMPT)
        display(Markdown(rows_to_markdown_table(rows)))
    else:
        print("Ollama is running but none of the candidate models are pulled yet.")
        print("Run: ollama pull qwen2.5:1.5b && ollama pull qwen2.5:3b")
else:
    print("Skipped — no Ollama server reachable. See note above.")

Skipped — no Ollama server reachable. See note above.


## 4. Take the lab scripts to the command line

For the full labs (including the long-prompt stress test and the failure-mode
analysis, which produce long output better read as files), run from the repo root:

```bash
uv run python scripts/module_01/lab_1_1_multi_model_run.py > reports/lab_1_1_raw_output.md
uv run python scripts/module_01/lab_1_2_long_prompt_stress_test.py --model qwen2.5:3b > reports/lab_1_2_raw_output.md
uv run python scripts/module_01/lab_1_3_small_model_failure_analysis.py --model qwen2.5:1.5b > reports/lab_1_3_raw_output.md
```

Then fold the results into `reports/module_01_local_llm_observations.md`, the module's
required deliverable, and manually annotate the failure modes in Lab 1.3's output.